In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, Dense, GlobalAveragePooling1D

# Step 1: load the dataset
df = pd.read_csv("/content/smartcart_customer_queries (3).csv")
print(df.shape)
print(df['category'].value_counts())

# Step 2: turn each query's text into a fixed-length list of numbers
MAX_LEN = 15
vectorizer = TextVectorization(output_mode="int", output_sequence_length=MAX_LEN)
vectorizer.adapt(df['query_text'])
X = vectorizer(df['query_text']).numpy()

# Step 3: turn each category into a number
labels = df['category'].astype('category')
y = labels.cat.codes.to_numpy()
class_names = labels.cat.categories
vocab_size = vectorizer.vocabulary_size()

print("Vocabulary size:", vocab_size)
print("Categories:", list(class_names))

(420, 3)
category
Apparel        140
Grocery        140
Electronics    140
Name: count, dtype: int64
Vocabulary size: 209
Categories: ['Apparel', 'Electronics', 'Grocery']


In [7]:
X = vectorizer(df['query_text']).numpy()

In [11]:
print("Available tokens:", tokens)
print("Vocabulary:", vectorizer.get_vocabulary())

Available tokens: [np.str_('kids'), np.str_('dress'), '', '', '', '', '', '', '', '', '', '', '', '', '']
Vocabulary: ['', '[UNK]', np.str_('size'), np.str_('the'), np.str_('free'), np.str_('under'), np.str_('shirt'), np.str_('but'), np.str_('is'), np.str_('quality'), np.str_('organic'), np.str_('waterproof'), np.str_('powder'), np.str_('milk'), np.str_('kids'), np.str_('oil'), np.str_('formal'), np.str_('cotton'), np.str_('and'), np.str_('very'), np.str_('shoes'), np.str_('almonds'), np.str_('8'), np.str_('wear'), np.str_('was'), np.str_('party'), np.str_('headphones'), np.str_('gown'), np.str_('good'), np.str_('for'), np.str_('cheap'), np.str_('pack'), np.str_('fit'), np.str_('wireless'), np.str_('sugar'), np.str_('smart'), np.str_('hoodie'), np.str_('honey'), np.str_('bulb'), np.str_('blue'), np.str_('winter'), np.str_('mouse'), np.str_('jacket'), np.str_('gaming'), np.str_('dress'), np.str_('charger'), np.str_('cable'), np.str_('spices'), np.str_('sneakers'), np.str_('price'), np.s

In [8]:
# Step 4: Select one sample query
sample_text = df['query_text'].iloc[0]

# Convert text into sequence of numbers
sample_seq = vectorizer(tf.constant([sample_text]))

# Convert sequence numbers into words
vocab = vectorizer.get_vocabulary()
tokens = [vocab[i] for i in sample_seq.numpy()[0]]

print("Sample Text:", sample_text)
print("Sample Sequence:", sample_seq.numpy())
print("Tokens:", tokens)

Sample Text: kids dress  
Sample Sequence: [[14 44  0  0  0  0  0  0  0  0  0  0  0  0  0]]
Tokens: [np.str_('kids'), np.str_('dress'), '', '', '', '', '', '', '', '', '', '', '', '', '']


In [9]:
sample_embeddings = embed_layer(sample_seq)

In [2]:
# Step 1: a tiny model -- Embedding, then built-in Attention, then classify
inputs = keras.Input(shape=(MAX_LEN,))
embed = Embedding(input_dim=vocab_size, output_dim=16)(inputs)

attention_output, attention_scores = tf.keras.layers.Attention()(
    [embed, embed], return_attention_scores=True
)
pooled = GlobalAveragePooling1D()(attention_output)
outputs = Dense(len(class_names), activation="softmax")(pooled)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [13]:
# Step 1: no mask -> BERT-style (every word sees every other word)
attn_layer = tf.keras.layers.Attention()
embed_layer = model.get_layer(index=1)   # reuse the trained Embedding layer
sample_embeddings = embed_layer(sample_seq)

_, weights_bert = attn_layer([sample_embeddings, sample_embeddings],
                              return_attention_scores=True)

# Step 2: causal mask -> GPT-style (each word sees only itself and earlier words)
_, weights_gpt = attn_layer([sample_embeddings, sample_embeddings],
                             return_attention_scores=True, use_causal_mask=True)

weights_bert = weights_bert.numpy()[0]
weights_gpt = weights_gpt.numpy()[0]

if "jacket" in tokens:
    target_word = "jacket"
else:
    target_word = next(
        (word for word in tokens if word not in ["", "[UNK]"]),
        None
    )

if target_word is None:
    raise ValueError("No valid word found in sample tokens")

jacket_idx = tokens.index(target_word)

print("Selected word:", target_word)
print("Selected index:", jacket_idx)

print(f"{'word':10s} {'BERT':>7s} {'GPT':>7s}")
for i, t in enumerate(tokens):
    print(f"{t:10s} {weights_bert[jacket_idx, i]:7.3f} {weights_gpt[jacket_idx, i]:7.3f}")

print(f"\nWords visible to 'jacket': BERT sees all {len(tokens)}; GPT sees only {jacket_idx + 1}")

Selected word: kids
Selected index: 0
word          BERT     GPT
kids         0.067   1.000
dress        0.066   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000
             0.067   0.000

Words visible to 'jacket': BERT sees all 15; GPT sees only 1


In [14]:
from transformers import pipeline

# BERT's pretraining task -- fill in a masked word
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
masked_query = "not satisfied with the winter [MASK], feels cheap and low quality"
for r in fill_mask(masked_query)[:3]:
    print(f"{r['token_str']:>10s}  (score={r['score']:.3f})")

# GPT-2's pretraining task -- continue text one word at a time
generator = pipeline("text-generation", model="gpt2")
prompt = "Customer: The winter jacket I bought feels cheap.\nAssistant:"
result = generator(prompt, max_new_tokens=30, num_return_sequences=1)
print(result[0]['generated_text'])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (9 outputs left out)

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Customer: The winter jacket I bought feels cheap.
Assistant: I wanted a winter jacket for my family. So I bought this jacket. I love it!
Assistant: A great fit.
Assistant: I
